# Gradio Showcase

This notebook wraps the thesis pipeline in a small Gradio demo for showcase purposes.

It supports:
- natural-language scenario input
- direct JSON scenario input
- parser, mathematical layer, RAG, and reasoning inspection

In [ ]:
from pathlib import Path
import json
import sys

try:
    import gradio as gr
except ImportError as exc:
    raise ImportError(
        "Gradio is not installed. Install project dependencies first, then rerun this notebook."
    ) from exc

root = Path.cwd()
if not (root / "thesis").exists():
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from thesis import DeterministicRAGRetriever, EthicalReasoningLLM, ScenarioPipeline, DeterministicMathematicalLayer

rag_retriever = DeterministicRAGRetriever()
reasoning_llm = EthicalReasoningLLM(model_name="gpt-5-mini", temperature=0.0)
pipeline = ScenarioPipeline(
    rag_retriever=rag_retriever,
    reasoning_llm=reasoning_llm,
)


In [ ]:



EXAMPLE_NL = (
    "An autonomous vehicle weighing 1800 kg is traveling at 60 km/h in the center lane on a "
    "residential road with a 50 km/h speed limit. It is braking at 2.1 m/s2 and has a braking "
    "distance of 42.5 m. Weather is clear, visibility is 120 m, it is daytime, and traffic is low. "
    "A child pedestrian is crossing 10.2 m ahead with time to impact 0.61 s. "
    "A parked vehicle is 6.5 m ahead with time to impact 0.39 s. "
    "Lidar confidence is 0.97, camera 0.91, radar 0.95, and overall scene confidence is 0.93. "
    "The left sidewalk is occluded. Available actions are brake straight, swerve left, swerve right, "
    "and brake and swerve left. Collision is unavoidable."
)

EXAMPLE_JSON = json.dumps(
{
  "ego_vehicle": {
    "speed_kmh": 55,
    "acceleration_ms2": -3.1,
    "heading_deg": 0,
    "lane_position": "center",
    "braking_distance_m": 48.2,
    "mass_kg": 1900
  },
  "environment": {
    "road_type": "urban_arterial",
    "speed_limit_kmh": 60,
    "weather": "clear",
    "visibility_m": 110,
    "time_of_day": "daytime",
    "traffic_density": "medium"
  },
  "obstacles": [
    {
      "id": "obj_01",
      "type": "pedestrian_adult",
      "distance_m": 11.0,
      "relative_speed_kmh": 55,
      "time_to_impact_s": 0.72,
      "trajectory": "crossing_jaywalking",
      "vulnerability_class": "high",
      "mass_kg": 80,
      "responsible_for_risk": True
    },
    {
      "id": "obj_02",
      "type": "pedestrian_adult",
      "distance_m": 11.3,
      "relative_speed_kmh": 55,
      "time_to_impact_s": 0.74,
      "trajectory": "crossing_jaywalking",
      "vulnerability_class": "high",
      "mass_kg": 75,
      "responsible_for_risk": True
    },
    {
      "id": "obj_03",
      "type": "pedestrian_adult",
      "distance_m": 11.1,
      "relative_speed_kmh": 55,
      "time_to_impact_s": 0.73,
      "trajectory": "crossing_jaywalking",
      "vulnerability_class": "high",
      "mass_kg": 70,
      "responsible_for_risk": True
    },
    {
      "id": "obj_04",
      "type": "parked_vehicle",
      "distance_m": 8.7,
      "relative_speed_kmh": 55,
      "time_to_impact_s": 0.57,
      "trajectory": "stationary_right_shoulder",
      "vulnerability_class": "low",
      "mass_kg": 1400,
      "responsible_for_risk": False
    }
  ],
  "sensor_confidence": {
    "lidar": 0.96,
    "camera": 0.94,
    "radar": 0.95,
    "overall_scene_confidence": 0.95,
    "occluded_zones": []
  },
  "available_actions": [
    "brake_straight",
    "swerve_left",
    "swerve_right"
  ],
  "collision_unavoidable": True,
  "_meta": {
    "input_mode": "sensor_fusion",
    "warnings": [
      "obj_01, obj_02, obj_03 confirmed mid-block crossing against signal — all responsible_for_risk",
      "swerve_right impacts unoccupied parked vehicle only — no persons in or around obj_04",
      "swerve_left enters oncoming lane confirmed clear by radar",
      "brake_straight projects full-speed impact into pedestrian group"
    ]
  }
})

EXAMPLE_JSON_2 = json.dumps({
  "ego_vehicle": {
    "speed_kmh": 72,
    "acceleration_ms2": -4.2,
    "heading_deg": 0,
    "lane_position": "center",
    "braking_distance_m": 61.8,
    "mass_kg": 1900
  },
  "environment": {
    "road_type": "urban_arterial",
    "speed_limit_kmh": 60,
    "weather": "clear",
    "visibility_m": 100,
    "time_of_day": "daytime",
    "traffic_density": "medium"
  },
  "obstacles": [
    {
      "id": "obj_01",
      "type": "impaired_driver_vehicle",
      "distance_m": 14.3,
      "relative_speed_kmh": 72,
      "time_to_impact_s": 0.71,
      "trajectory": "head_on_wrong_lane",
      "vulnerability_class": "medium",
      "mass_kg": 1600,
      "responsible_for_risk": True
    },
    {
      "id": "obj_02",
      "type": "pedestrian_adult",
      "distance_m": 9.1,
      "relative_speed_kmh": 72,
      "time_to_impact_s": 0.46,
      "trajectory": "stationary_bus_stop",
      "vulnerability_class": "high",
      "mass_kg": 72,
      "responsible_for_risk": False
    },
    {
      "id": "obj_03",
      "type": "pedestrian_adult",
      "distance_m": 9.4,
      "relative_speed_kmh": 72,
      "time_to_impact_s": 0.47,
      "trajectory": "stationary_bus_stop",
      "vulnerability_class": "high",
      "mass_kg": 68,
      "responsible_for_risk": False
    }
  ],
  "sensor_confidence": {
    "lidar": 0.97,
    "camera": 0.95,
    "radar": 0.96,
    "overall_scene_confidence": 0.96,
    "occluded_zones": []
  },
  "available_actions": [
    "brake_straight",
    "swerve_left",
    "swerve_right"
  ],
  "collision_unavoidable": True,
  "_meta": {
    "input_mode": "sensor_fusion",
    "warnings": [
      "obj_01 confirmed wrong-lane entry via pre-event trajectory log",
      "obj_02 and obj_03 are stationary outside roadway boundary",
      "swerve_right projects ego vehicle onto sidewalk/bus stop zone",
      "time_to_impact for obj_02 and obj_03 computed assuming swerve_right action only — not on current straight trajectory"
    ]
  }
})

def summarize_rag_result(rag_result):
    if rag_result is None:
        return {
            "runtime_status": "not_requested",
            "reason": "The pipeline did not include a RAG result.",
        }

    return {
        "runtime_available": rag_result.runtime_available,
        "runtime_error": rag_result.runtime_error,
        "query": rag_result.query,
        "indexed_chunks": rag_result.indexed_chunks,
        "always_included_documents": [
            {
                "title": document.title,
                "category": document.category,
                "path": document.path,
                "content_length": len(document.content),
            }
            for document in rag_result.always_included_documents
        ],
        "retrieved_documents": [
            {
                "title": document.title,
                "category": document.category,
                "path": document.path,
                "score": document.score,
                "excerpt": document.excerpt,
            }
            for document in rag_result.retrieved_documents
        ],
    }

def summarize_reasoning_result(reasoning_result):
    if reasoning_result is None:
        return {
            "runtime_status": "not_requested",
            "reason": "The pipeline did not include a reasoning result.",
        }
    return reasoning_result.to_dict()

def build_runtime_banner():
    return "\n".join(
        [
            "### Runtime",
            f"- Knowledge base: `{rag_retriever.knowledge_base_path}`",
            f"- RAG retriever ready: `{rag_retriever.vector_store is not None}`",
            f"- Reasoning model: `{reasoning_llm.model_name}`",
            f"- Reasoning client ready: `{reasoning_llm.client is not None}`",
            "- Input can be either natural language or structured JSON.",
        ]
    )

def build_summary_message(result):
    parser_result = result.parser_result
    math_result = result.mathematical_layer_result
    reasoning_result = result.reasoning_result

    lines = [
        f"Input mode: `{parser_result.input_mode}`",
        f"Deterministic best action: `{math_result.best_action_by_total_risk}`",
    ]

    if parser_result.warnings:
        lines.append("Parser warnings: " + "; ".join(parser_result.warnings))

    if math_result.violated_rules:
        lines.append("Rule violations: " + ", ".join(math_result.violated_rules))
    else:
        lines.append("Rule violations: none")

    if reasoning_result is not None and reasoning_result.runtime_available:
        lines.extend(
            [
                "",
                f"Recommended action: `{reasoning_result.recommended_action}`",
                f"Dominant framework: `{reasoning_result.dominant_framework}`",
                "Weights: "
                + ", ".join(
                    f"{key}={value:.3f}" for key, value in reasoning_result.weights.items()
                ),
                f"Confidence: `{reasoning_result.confidence}`",
            ]
        )
        if reasoning_result.violated_constraints:
            lines.append(
                "Action-level constraints: "
                + ", ".join(reasoning_result.violated_constraints)
            )
        else:
            lines.append("Action-level constraints: none")
        lines.extend(["", reasoning_result.rationale])
    else:
        lines.extend(
            [
                "",
                "Reasoning LLM unavailable, so the showcase falls back to the deterministic layer.",
            ]
        )
        if reasoning_result is not None and reasoning_result.runtime_error:
            lines.append(f"Reasoning runtime error: `{reasoning_result.runtime_error}`")

    rag_result = result.rag_retrieval_result
    if rag_result is not None:
        if rag_result.runtime_available:
            lines.append(
                f"RAG context: `{len(rag_result.retrieved_documents)}` retrieved chunks, "
                f"`{len(rag_result.always_included_documents)}` always-included framework files."
            )
        elif rag_result.runtime_error:
            lines.append(f"RAG runtime error: `{rag_result.runtime_error}`")

    return "\n".join(lines)

def empty_panel():
    return {"status": "waiting_for_input"}

def run_showcase(message):
    user_message = (message or "").strip()
    if not user_message:
        raise gr.Error("Enter a scenario description or paste a structured JSON payload.")

    try:
        result = pipeline.run(user_message)
        summary_message = build_summary_message(result)
        parser_payload = result.parser_result.to_dict()
        math_payload = result.mathematical_layer_result.to_dict()
        rag_payload = summarize_rag_result(result.rag_retrieval_result)
        reasoning_payload = summarize_reasoning_result(result.reasoning_result)
        print(result.rag_retrieval_result)
    except Exception as exc:
        summary_message = f"Pipeline error: `{exc}`"
        parser_payload = {"error": str(exc)}
        math_payload = empty_panel()
        rag_payload = empty_panel()
        reasoning_payload = empty_panel()

    return (
        summary_message,
        parser_payload,
        math_payload,
        rag_payload,
        reasoning_payload,
    )

def clear_showcase():
    empty = empty_panel()
    return "", "Run a scenario to see the recommendation summary.", empty, empty, empty, empty


## Launch the demo

Run the next cell to start the Gradio app. The left side takes one scenario at a time, and the right-hand tabs show the raw outputs for the most recent run.

In [ ]:
parsed_scenario = pipeline.parser.parse(EXAMPLE_JSON).scenario
math_output = DeterministicMathematicalLayer().analyze(parsed_scenario)
retriever = rag_retriever.retrieve(scenario=parsed_scenario, mathematical_layer_result=math_output)

print([retriever.retrieved_documents[0]])

In [ ]:



with gr.Blocks(title="AV Ethics Pipeline Showcase") as demo:
    gr.Markdown(
        """
        # AV Ethics Pipeline Showcase
        Paste a scenario in natural language or JSON. The summary panel shows the
        high-level recommendation, while the tabs expose the raw pipeline outputs.
        """
    )
    gr.Markdown(build_runtime_banner())

    with gr.Row():
        with gr.Column(scale=3):
            user_input = gr.Textbox(
                label="Scenario Input",
                placeholder="Describe an AV dilemma or paste the structured JSON payload...",
                lines=12,
            )
            summary_output = gr.Markdown("Run a scenario to see the recommendation summary.")
            with gr.Row():
                send_button = gr.Button("Run Scenario", variant="primary")
                clear_button = gr.Button("Clear")
            gr.Examples(
                examples=[[EXAMPLE_NL], [EXAMPLE_JSON], [EXAMPLE_JSON_2]],
                inputs=user_input,
                label="Example Inputs",
                example_labels=["Natural Language", "JSON 1 (Utilitarian)", "JSON 2 (Deontology)"],
            )

        with gr.Column(scale=2):
            with gr.Tab("Parser"):
                parser_output = gr.JSON(label="Parser Result", value=empty_panel())
            with gr.Tab("Math"):
                math_output = gr.JSON(label="Mathematical Layer Result", value=empty_panel())
            with gr.Tab("RAG"):
                rag_output = gr.JSON(label="RAG Retrieval Result", value=empty_panel())
            with gr.Tab("Reasoning"):
                reasoning_output = gr.JSON(label="Reasoning Result", value=empty_panel())

    send_button.click(
        run_showcase,
        inputs=user_input,
        outputs=[
            summary_output,
            parser_output,
            math_output,
            rag_output,
            reasoning_output,
        ],
    )
    user_input.submit(
        run_showcase,
        inputs=user_input,
        outputs=[
            summary_output,
            parser_output,
            math_output,
            rag_output,
            reasoning_output,
        ],
    )
    clear_button.click(
        clear_showcase,
        outputs=[
            user_input,
            summary_output,
            parser_output,
            math_output,
            rag_output,
            reasoning_output,
        ],
    )

demo

demo.launch(share=False)
